# tool_02 提交调度

## 功能
1. **生成提交文件**：job_list.txt、submit 脚本
2. **核数配给**：按分区方案分配
3. **Hub 分区**：young.ng（Slurm）、young（SGE）

## 与频率计算一致
提交前检查：POSCAR、POTCAR、KPOINTS、INCAR 齐全

In [ ]:
import os
from pathlib import Path

In [ ]:
import sys
_p = Path.cwd()
while _p != _p.parent:
    if (_p / "shared" / "config.py").exists():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
from shared.load_config import load_config
cfg = load_config()
go_root = cfg.GO_ROOT
DEFAULT_PARTITION = getattr(cfg, 'DEFAULT_PARTITION', 'young')

In [ ]:
HUB_PARTITIONS = {
    "young_ng": {
        "scheduler": "slurm",
        "nodes": 4,
        "tasks_per_node": 32,
        "total_cores": 128,
        "module": "module load vasp/5.4.4-intel-oneapi-mpi/oneapi-2024.2.1",
        "mpi_cmd": "srun vasp_std",
    },
    "young": {
        "scheduler": "sge",
        "total_cores": 128,
        "pe": "mpi",
        "module": "module load vasp/5.4.4-18apr2017-vtst-r178/intel-2017-update1",
        "mpi_cmd": "gerun vasp_std",
    },
}

In [ ]:
def scan_job_dirs(root):
    """扫描含 POSCAR+INCAR+KPOINTS+POTCAR 的任务目录。"""
    job_dirs = []
    r = Path(root)
    if not r.exists():
        return []
    for sub in r.iterdir():
        if sub.is_dir() and (sub / "POSCAR").exists() and (sub / "INCAR").exists():
            if (sub / "KPOINTS").exists() and (sub / "POTCAR").exists():
                job_dirs.append(str(sub))
    return sorted(job_dirs)


def gen_sge_submit(job_dir, job_name, partition="young"):
    cfg = HUB_PARTITIONS.get(partition, HUB_PARTITIONS["young"])
    cores = cfg["total_cores"]
    pe = cfg.get("pe", "mpi")
    return f"""#!/bin/bash
#$ -N {job_name}
#$ -S /bin/bash
#$ -l h_rt=24:00:00
#$ -P Gold
#$ -A UCL_chemM_Catlow
#$ -pe {pe} {cores}
#$ -cwd

source /etc/profile.d/modules.sh
{cfg['module']}

cd {job_dir}
{cfg['mpi_cmd']} > result.$JOB_ID
"""


def gen_slurm_submit(job_dir, job_name, partition="young_ng"):
    cfg = HUB_PARTITIONS.get(partition, HUB_PARTITIONS["young_ng"])
    nodes = cfg.get("nodes", 4)
    tpn = cfg.get("tasks_per_node", 32)
    return f"""#!/bin/bash
#SBATCH -J {job_name}
#SBATCH -N {nodes}
#SBATCH --ntasks-per-node={tpn}
#SBATCH -t 24:00:00
#SBATCH -p young

source /etc/profile.d/modules.sh
{cfg['module']}

cd {job_dir}
{cfg['mpi_cmd']} > result.$SLURM_JOB_ID
"""


def gen_array_submit_sge(job_list_path, job_dirs, partition="young"):
    cfg = HUB_PARTITIONS.get(partition, HUB_PARTITIONS["young"])
    cores = cfg["total_cores"]
    pe = cfg.get("pe", "mpi")
    n = len(job_dirs)
    return f"""#!/bin/bash
#$ -N GO_abs
#$ -S /bin/bash
#$ -l h_rt=24:00:00
#$ -P Gold
#$ -A UCL_chemM_Catlow
#$ -pe {pe} {cores}
#$ -t 1-{n}
#$ -cwd

source /etc/profile.d/modules.sh
{cfg['module']}

JOB_LIST="{job_list_path}"
JOB_DIR=$(sed -n "${{SGE_TASK_ID}}p" "$JOB_LIST")
JOB_NAME=$(basename "$JOB_DIR")
cd "$JOB_DIR"
{cfg['mpi_cmd']} > result.$JOB_ID.$SGE_TASK_ID.$JOB_NAME
"""


def gen_array_submit_slurm(job_list_path, job_dirs, partition="young_ng"):
    cfg = HUB_PARTITIONS.get(partition, HUB_PARTITIONS["young_ng"])
    nodes = cfg.get("nodes", 4)
    tpn = cfg.get("tasks_per_node", 32)
    n = len(job_dirs)
    return f"""#!/bin/bash
#SBATCH -J GO_abs
#SBATCH -N {nodes}
#SBATCH --ntasks-per-node={tpn}
#SBATCH -t 24:00:00
#SBATCH -p young
#SBATCH --array=1-{n}

source /etc/profile.d/modules.sh
{cfg['module']}

JOB_LIST="{job_list_path}"
JOB_DIR=$(sed -n "${{SLURM_ARRAY_TASK_ID}}p" "$JOB_LIST")
JOB_NAME=$(basename "$JOB_DIR")
cd "$JOB_DIR"
{cfg['mpi_cmd']} > result.$SLURM_JOB_ID.$SLURM_ARRAY_TASK_ID.$JOB_NAME
"""

In [ ]:
# 提交前检查：POSCAR、POTCAR、KPOINTS、INCAR 齐全
from shared.pre_check import run_pre_check
if not run_pre_check(go_root):
    raise RuntimeError("提交前检查未通过，详见 shared/移交检查.md")

In [ ]:
job_dirs = scan_job_dirs(go_root)
print(f"📌 扫描到 {len(job_dirs)} 个任务目录")
for d in job_dirs:
    print(f"  - {os.path.relpath(d, go_root)}")

In [ ]:
job_list_path = os.path.join(go_root, "job_list.txt")
with open(job_list_path, "w") as f:
    for d in job_dirs:
        f.write(os.path.abspath(d) + "\n")
print(f"✔ job_list.txt: {job_list_path}")

partition = DEFAULT_PARTITION
cfg = HUB_PARTITIONS.get(partition, HUB_PARTITIONS["young"])
sched = cfg["scheduler"]

if sched == "sge":
    submit_content = gen_array_submit_sge(os.path.abspath(job_list_path), job_dirs, partition)
    submit_path = os.path.join(go_root, "submit.sh")
    with open(submit_path, "w") as f:
        f.write(submit_content)
    print(f"✔ submit.sh (SGE 数组): {submit_path}")
    print(f"  分区: {partition}, 每任务 {cfg['total_cores']} 核")
    print(f"\n>>> 提交: cd {go_root} && qsub submit.sh")
else:
    submit_content = gen_array_submit_slurm(os.path.abspath(job_list_path), job_dirs, partition)
    submit_path = os.path.join(go_root, "submit.sh")
    with open(submit_path, "w") as f:
        f.write(submit_content)
    print(f"✔ submit.sh (Slurm 数组): {submit_path}")
    print(f"  分区: {partition}")
    print(f"\n>>> 提交: cd {go_root} && sbatch submit.sh")